In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import openslide


def extract_he_tissue_contour_image(
    slide_path,
    max_dim=1800,
    sat_thresh=8,
    val_thresh=253,
    od_thresh=0.025,
    min_area=500,
    open_kernel_size=3,
    close_kernel_size=45,
    grow_pixels=8,
    use_convex_hull=False,
    show_original=True,
    show_result=True,
    debug=False,
    return_mask=False,
    return_info=False
):
    """
    Extrae el tejido de una H&E tipo .mrxs usando el contorno externo del tejido.

    Por defecto devuelve solo:
        tissue_only_rgb

    Opcionalmente puede devolver:
        mask_tissue
        info

    Parameters
    ----------
    slide_path : str or Path
        Ruta al archivo .mrxs.

    max_dim : int
        Tamaño máximo del lado largo usado para leer la imagen a baja resolución.

    sat_thresh : int
        Umbral de saturación HSV. Más bajo detecta tejido más pálido.

    val_thresh : int
        Umbral de brillo HSV. Más alto permite zonas más claras.

    od_thresh : float
        Umbral de optical density. Más bajo detecta tejido muy claro.

    min_area : int
        Área mínima de componentes a conservar.

    open_kernel_size : int
        Kernel para eliminar ruido pequeño.

    close_kernel_size : int
        Kernel para cerrar huecos y unir zonas del tejido.

    grow_pixels : int
        Dilatación final del contorno para no cortar tejido.

    use_convex_hull : bool
        Si True, usa convex hull. Es más conservador, puede meter más fondo.

    show_original : bool
        Plotea la H&E original leída.

    show_result : bool
        Plotea el resultado final.

    debug : bool
        Muestra pasos intermedios.

    return_mask : bool
        Si True, devuelve también la máscara final.

    return_info : bool
        Si True, devuelve también información auxiliar.
    """

    slide_path = Path(slide_path)

    if not slide_path.exists():
        raise FileNotFoundError(f"No existe el archivo: {slide_path}")

    # ============================================================
    # 1. Cargar slide a baja resolución
    # ============================================================
    slide = openslide.OpenSlide(str(slide_path))
    level_dims = slide.level_dimensions

    chosen_level = None
    for i, (w, h) in enumerate(level_dims):
        if max(w, h) <= max_dim:
            chosen_level = i
            break

    if chosen_level is None:
        chosen_level = len(level_dims) - 1

    level_w, level_h = level_dims[chosen_level]

    rgb_pil = slide.read_region(
        location=(0, 0),
        level=chosen_level,
        size=(level_w, level_h)
    ).convert("RGB")

    slide.close()

    rgb_u8 = np.array(rgb_pil, dtype=np.uint8)

    # ============================================================
    # 2. Mapas HSV y optical density
    # ============================================================
    hsv = cv2.cvtColor(rgb_u8, cv2.COLOR_RGB2HSV)

    S = hsv[:, :, 1]
    V = hsv[:, :, 2]

    rgb_float = rgb_u8.astype(np.float32)
    od = -np.log((rgb_float + 1.0) / 255.0)
    od_sum = od.sum(axis=2)

    # ============================================================
    # 3. Máscara inicial: tejido teñido o tejido pálido
    # ============================================================
    mask_sat = (
        (S > sat_thresh) &
        (V < val_thresh) &
        (V > 20)
    ).astype(np.uint8) * 255

    mask_od = (
        (od_sum > od_thresh) &
        (V < val_thresh) &
        (V > 20)
    ).astype(np.uint8) * 255

    mask_raw = cv2.bitwise_or(mask_sat, mask_od)

    # ============================================================
    # 4. Morfología
    # ============================================================
    mask_morph = mask_raw.copy()

    if open_kernel_size > 0:
        kernel_open = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (open_kernel_size, open_kernel_size)
        )
        mask_morph = cv2.morphologyEx(mask_morph, cv2.MORPH_OPEN, kernel_open)

    if close_kernel_size > 0:
        kernel_close = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (close_kernel_size, close_kernel_size)
        )
        mask_morph = cv2.morphologyEx(mask_morph, cv2.MORPH_CLOSE, kernel_close)

    # ============================================================
    # 5. Componentes conectados
    # ============================================================
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        mask_morph,
        connectivity=8
    )

    mask_components = np.zeros_like(mask_morph, dtype=np.uint8)

    for label_id in range(1, num_labels):
        area = stats[label_id, cv2.CC_STAT_AREA]
        if area >= min_area:
            mask_components[labels == label_id] = 255

    if np.count_nonzero(mask_components) == 0:
        raise ValueError(
            "No se detectó tejido. Prueba sat_thresh=5, od_thresh=0.012, val_thresh=255."
        )

    # ============================================================
    # 6. Extraer contorno externo y rellenarlo
    # ============================================================
    contours, _ = cv2.findContours(
        mask_components,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    if len(contours) == 0:
        raise ValueError("No se encontraron contornos de tejido.")

    # Normalmente el tejido principal es el mayor contorno
    largest_contour = max(contours, key=cv2.contourArea)

    mask_contour = np.zeros_like(mask_components, dtype=np.uint8)

    if use_convex_hull:
        hull = cv2.convexHull(largest_contour)
        cv2.drawContours(mask_contour, [hull], -1, 255, thickness=cv2.FILLED)
    else:
        cv2.drawContours(mask_contour, [largest_contour], -1, 255, thickness=cv2.FILLED)

    # Dilatar ligeramente para no cortar borde
    mask_final = mask_contour.copy()

    if grow_pixels > 0:
        kernel_grow = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (2 * grow_pixels + 1, 2 * grow_pixels + 1)
        )
        mask_final = cv2.dilate(mask_final, kernel_grow, iterations=1)

    # ============================================================
    # 7. Aplicar máscara: tejido visible, resto negro
    # ============================================================
    tissue_only_rgb = rgb_u8.copy()
    tissue_only_rgb[mask_final == 0] = 0

    # ============================================================
    # 8. Plots
    # ============================================================
    if debug:
        od_vis = (od_sum - od_sum.min()) / (od_sum.max() - od_sum.min() + 1e-8)

        plt.figure(figsize=(18, 10))

        plt.subplot(2, 4, 1)
        plt.imshow(rgb_u8)
        plt.title(f"H&E original | level={chosen_level}")
        plt.axis("off")

        plt.subplot(2, 4, 2)
        plt.imshow(S, cmap="gray")
        plt.title("Saturación HSV")
        plt.axis("off")

        plt.subplot(2, 4, 3)
        plt.imshow(od_vis, cmap="gray")
        plt.title("Optical density")
        plt.axis("off")

        plt.subplot(2, 4, 4)
        plt.imshow(mask_raw, cmap="gray")
        plt.title("Máscara inicial")
        plt.axis("off")

        plt.subplot(2, 4, 5)
        plt.imshow(mask_morph, cmap="gray")
        plt.title("Después morfología")
        plt.axis("off")

        plt.subplot(2, 4, 6)
        plt.imshow(mask_components, cmap="gray")
        plt.title("Componentes filtrados")
        plt.axis("off")

        plt.subplot(2, 4, 7)
        plt.imshow(mask_final, cmap="gray")
        plt.title("Máscara final por contorno")
        plt.axis("off")

        plt.subplot(2, 4, 8)
        plt.imshow(tissue_only_rgb)
        plt.title("Solo tejido")
        plt.axis("off")

        plt.tight_layout()
        plt.show()

    elif show_original and show_result:
        plt.figure(figsize=(10, 6))

        plt.subplot(1, 2, 1)
        plt.imshow(rgb_u8)
        plt.title(f"H&E original | level={chosen_level}")
        plt.axis("off")

        plt.subplot(1, 2, 2)
        plt.imshow(tissue_only_rgb)
        plt.title("Solo tejido")
        plt.axis("off")

        plt.tight_layout()
        plt.show()

    elif show_original:
        plt.figure(figsize=(6, 6))
        plt.imshow(rgb_u8)
        plt.title(f"H&E original | level={chosen_level}")
        plt.axis("off")
        plt.show()

    elif show_result:
        plt.figure(figsize=(6, 6))
        plt.imshow(tissue_only_rgb)
        plt.title("Solo tejido")
        plt.axis("off")
        plt.show()

    # ============================================================
    # 9. Return limpio
    # ============================================================
    outputs = [tissue_only_rgb]

    if return_mask:
        outputs.append(mask_final)

    if return_info:
        info = {
            "chosen_level": chosen_level,
            "read_dimensions": (level_w, level_h),
            "sat_thresh": sat_thresh,
            "val_thresh": val_thresh,
            "od_thresh": od_thresh,
            "grow_pixels": grow_pixels,
            "use_convex_hull": use_convex_hull,
            "mask_area_px": int(np.count_nonzero(mask_final)),
        }
        outputs.append(info)

    if len(outputs) == 1:
        return tissue_only_rgb

    return tuple(outputs)

In [ ]:
slide_path = Path(r"Datos\SB013\SB013\H&E_python_EDU_creado\SB013.mrxs")

tissue_only_rgb = extract_he_tissue_contour_image(
    slide_path,
    max_dim=1800,
    show_original=True,
    show_result=True,
    debug=False
)